# learning_resources 데이터셋 정제 및 EDA

## 1. 데이터 로드 및 기본 정보 확인

In [1]:
import pandas as pd
from IPython.display import display

# 1. ??? ?? ? ?? ?? ??
df = pd.read_csv("learning_resources.csv", encoding="utf-8-sig")

shape_df = pd.DataFrame({
    "metric": ["rows", "columns"],
    "value": [df.shape[0], df.shape[1]],
})

schema_df = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null": df.notna().sum().values,
    "null_count": df.isna().sum().values,
    "null_ratio(%)": (df.isna().mean() * 100).round(2).values,
    "nunique": df.nunique(dropna=True).values,
})

memory_df = pd.DataFrame({
    "metric": ["memory_usage_kb", "memory_usage_mb"],
    "value": [
        round(df.memory_usage(deep=True).sum() / 1024, 2),
        round(df.memory_usage(deep=True).sum() / 1024 / 1024, 2),
    ],
})

print("[Shape]")
display(shape_df)

print("[Schema]")
display(schema_df)

print("[Memory]")
display(memory_df)

print("[Sample: top 3 rows]")
display(df.head(3))



[Shape]


,metric,value
0,rows,183
1,columns,16


[Schema]


,column,dtype,non_null,null_count,null_ratio(%),nunique
0,source_type,object,183,0,0.00,2
1,external_id,object,183,0,0.00,183
2,main_category,object,183,0,0.00,6
3,sub_category,object,183,0,0.00,30
4,title,object,183,0,0.00,179
5,description,object,158,25,13.66,157
6,provider_name,object,183,0,0.00,69
7,instructor_name,object,183,0,0.00,150
8,difficulty_level,object,183,0,0.00,4
9,semester,object,183,0,0.00,36


[Memory]


,metric,value
0,memory_usage_kb,388.41
1,memory_usage_mb,0.38


[Sample: top 3 rows]


,source_type,external_id,main_category,sub_category,title,description,provider_name,instructor_name,difficulty_level,semester,url,thumbnail_url,syllabus_url,content_type,view_count,popular_score
0,KOCW,607f5aaa31467d55,공학,컴퓨터공학,프로그래밍개론,본 교과목은 아두이노(Arduino) 기반의 피지컬 컴퓨팅을 구현하기 위한 기초 프...,국립목포대학교,박장현,beginner,2025년 1학기,http://www.kocw.net/home/cview.do?cid=607f5aaa...,http://www.kocw.net/home/common/contents/thumb...,http://www.kocw.net/home/common/contents/sylla...,video,8.0,NaN
1,KOCW,77b5b8f2d825bc0b,공학,정보통신공학,AI 리터러시,"인공지능(AI)의 기본 개념부터, 4차산업혁명과 AI, 컴퓨터와의 관계, 생성형AI...",국립목포대학교,이연우,intermediate,2025년 1학기,http://www.kocw.net/home/cview.do?cid=77b5b8f2...,http://www.kocw.net/home/common/contents/thumb...,http://www.kocw.net/home/common/contents/sylla...,video,12.0,NaN
2,KOCW,83e4d2e2e90b28c9,공학,메카트로닉스공학,기계제도및실습,본 강의는 기계설계를 위한 기초개념을 배우고 설계 결과를 도면으로 표현하는 기계제도...,건국대학교,강현규,beginner,2025년 1학기,http://www.kocw.net/home/cview.do?cid=83e4d2e2...,http://www.kocw.net/home/common/contents/thumb...,http://www.kocw.net/home/common/contents/sylla...,video,408.0,NaN


### 1-1. EDA 시작 템플릿 (요약 통계)
- 아래 표는 수치형/범주형 컬럼을 함께 빠르게 점검하기 위한 기본 요약입니다.
- 기존 `df`를 그대로 사용하며, 이후 결측치/이상치 분석의 기준점으로 활용합니다.


In [2]:
# 1-1. EDA 시작 템플릿: 전체 요약 통계 테이블
# NOTE: 반드시 이전 셀(데이터 로드 셀) 실행 후 사용하세요. df가 없으면 NameError가 발생합니다.

# pandas 버전에 따라 datetime_is_numeric 옵션이 미지원일 수 있어 사용하지 않습니다.
summary_df = df.describe(include="all").T

# 결측 지표 추가
summary_df["missing_count"] = df.isna().sum()
summary_df["missing_ratio(%)"] = (df.isna().mean() * 100).round(2)

# 자주 보는 지표를 앞쪽에 배치
priority_cols = [
    "missing_count", "missing_ratio(%)", "count", "unique", "top", "freq",
    "mean", "std", "min", "25%", "50%", "75%", "max"
]
existing_cols = [c for c in priority_cols if c in summary_df.columns]
remaining_cols = [c for c in summary_df.columns if c not in existing_cols]
summary_df = summary_df[existing_cols + remaining_cols]

# 정제 우선순위 확인을 위해 결측 비율 기준 정렬
summary_df = summary_df.sort_values(by="missing_ratio(%)", ascending=False)

from IPython.display import display
print("[EDA Starter: describe(include='all').T + missing stats]")
display(summary_df)


[EDA Starter: describe(include='all').T + missing stats]


,missing_count,missing_ratio(%),count,unique,top,freq,mean,std,min,25%,50%,75%,max
popular_score,176,96.17,7.0,NaN,NaN,NaN,4.857143,0.377964,4.0,5.0,5.0,5.0,5.0
syllabus_url,59,32.24,124,124,http://www.kocw.net/home/common/contents/sylla...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
view_count,56,30.60,127.0,NaN,NaN,NaN,3126.527559,2884.550596,8.0,1668.5,2624.0,3707.5,26595.0
description,25,13.66,158,157,"키워드: 인공지능,AI",2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sub_category,0,0.00,183,30,컴퓨터공학,46,NaN,NaN,NaN,NaN,NaN,NaN,NaN
source_type,0,0.00,183,2,KOCW,127,NaN,NaN,NaN,NaN,NaN,NaN,NaN
main_category,0,0.00,183,6,공학,165,NaN,NaN,NaN,NaN,NaN,NaN,NaN
external_id,0,0.00,183,183,607f5aaa31467d55,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
instructor_name,0,0.00,183,150,김대기,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
provider_name,0,0.00,183,69,영남대학교,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. 결측치 분석

In [ ]:
print("\n" + "=" * 60)
print("2. 결측치 분석")
print("=" * 60)

missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    "결측수": missing,
    "결측률(%)": missing_pct,
    "비결측수": len(df) - missing
}).sort_values("결측률(%)", ascending=False)

print("\n📉 컬럼별 결측치 현황:")
display(missing_df)

# 결측이 있는 컬럼만 필터
cols_with_missing = missing_df[missing_df["결측수"] > 0]
if len(cols_with_missing) > 0:
    print(f"\n⚠️  결측치가 있는 컬럼: {len(cols_with_missing)}개")
else:
    print(f"\n✅ 결측치 없음")



2. 결측치 분석

📉 컬럼별 결측치 현황:


,결측수,결측률(%),비결측수
popular_score,176,96.17,7
syllabus_url,59,32.24,124
view_count,56,30.60,127
description,25,13.66,158
sub_category,0,0.00,183
source_type,0,0.00,183
main_category,0,0.00,183
external_id,0,0.00,183
instructor_name,0,0.00,183
provider_name,0,0.00,183



⚠️  결측치가 있는 컬럼: 4개
               결측수  결측률(%)  비결측수
popular_score  176   96.17     7
syllabus_url    59   32.24   124
view_count      56   30.60   127
description     25   13.66   158


### 앞으로의 진행

1. `popular_score`, `syllabus_url`, `view_count` 필드는 높은 결측률, 상대적으로 서비스에서 낮은 비중을 가지므로 삭제하는 방향으로 진행

## 3. 데이터 중복 및 분포 분석

### 3-1. 중복 데이터 확인

In [15]:
print("\n" + "=" * 60)
print("3-1. 중복 데이터 확인")
print("=" * 60)

# 전체 행 중복
full_duplicates = df.duplicated().sum()
print(f"\n🔁 전체 행 중복: {full_duplicates}개")

# external_id 기준 중복
eid_duplicates = df.duplicated(subset=["external_id"]).sum()
print(f"🔁 external_id 중복: {eid_duplicates}개")

# source_type + external_id 복합키 중복
composite_duplicates = df.duplicated(subset=["source_type", "external_id"]).sum()
print(f"🔁 (source_type + external_id) 복합키 중복: {composite_duplicates}개")

if eid_duplicates > 0:
    dup_eids = df[df.duplicated(subset=["external_id"], keep=False)]
    print(f"\n 중복 external_id 샘플:")
    print(dup_eids[["source_type", "external_id", "title"]].head(10).to_string())


3-1. 중복 데이터 확인

🔁 전체 행 중복: 0개
🔁 external_id 중복: 0개
🔁 (source_type + external_id) 복합키 중복: 0개


### 3-2. 범주형 column 분포 분석

In [ ]:
print("\n" + "=" * 60)
print("3-2. 범주형 컬럼 분포 분석")
print("=" * 60)

categorical_cols = ["source_type", "main_category", "sub_category", 
                    "difficulty_level", "content_type"]

for col in categorical_cols:
    print(f"\n📊 [{col}] 분포 (고유값: {df[col].nunique()}개)")
    vc = df[col].value_counts(dropna=False)
    # 상위 15개만 출력
    for val, cnt in vc.items():
        pct = cnt / len(df) * 100
        val_display = val if pd.notna(val) else "(결측)"
        print(f"   {val_display}: {cnt}개 ({pct:.1f}%)")
    # if len(vc) > 15:
    #     print(f"   ... 외 {len(vc) - 15}개")


4. 범주형 컬럼 분포 분석

📊 [source_type] 분포 (고유값: 2개)
   KOCW: 127개 (69.4%)
   KMOOC: 56개 (30.6%)

📊 [main_category] 분포 (고유값: 6개)
   공학: 165개 (90.2%)
   사회: 9개 (4.9%)
   자연: 4개 (2.2%)
   예체능: 3개 (1.6%)
   인문: 1개 (0.5%)
   의약: 1개 (0.5%)

📊 [sub_category] 분포 (고유값: 30개)
   컴퓨터공학: 46개 (25.1%)
   소프트웨어공학: 33개 (18.0%)
   컴퓨터 · 통신: 33개 (18.0%)
   컴퓨터과학: 16개 (8.7%)
   정보통신공학: 6개 (3.3%)
   정보과학: 5개 (2.7%)
   전기 · 전자: 4개 (2.2%)
   사회과학: 4개 (2.2%)
   경영 · 경제: 4개 (2.2%)
   전자공학: 4개 (2.2%)
   메카트로닉스공학: 3개 (1.6%)
   산업공학: 2개 (1.1%)
   기계공학: 2개 (1.1%)
   멀티미디어학: 2개 (1.1%)
   디자인: 2개 (1.1%)
   수학 · 물리 · 천문 · 지리: 2개 (1.1%)
   전기전자공학: 2개 (1.1%)
   신소재공학: 1개 (0.5%)
   건축공학: 1개 (0.5%)
   화학공학: 1개 (0.5%)
   건축학: 1개 (0.5%)
   컴퓨터ㆍ통신: 1개 (0.5%)
   법률: 1개 (0.5%)
   인문과학: 1개 (0.5%)
   전기공학: 1개 (0.5%)
   교통 · 운송: 1개 (0.5%)
   생물 · 화학 · 환경: 1개 (0.5%)
   농림 · 수산: 1개 (0.5%)
   응용예술: 1개 (0.5%)
   치료 · 보건: 1개 (0.5%)

📊 [difficulty_level] 분포 (고유값: 4개)
   beginner: 65개 (35.5%)
   intermediate: 54개 (29.5%)
   unknown: 40개 (21

## 4. 이상치 및 데이터 품질 확인

In [ ]:
# 4. 이상치 및 데이터 품질 확인
# NOTE: 이전 셀에서 df가 먼저 로드되어 있어야 합니다. df가 없으면 NameError가 발생합니다.

from IPython.display import display

# description 길이 분포 확인
description_len = df["description"].fillna("").astype(str).str.len()
description_len_df = pd.DataFrame({
    "metric": ["count", "min", "25%", "50%", "75%", "max", "mean"],
    "value": [
        int(description_len.shape[0]),
        int(description_len.min()),
        round(description_len.quantile(0.25), 2),
        round(description_len.quantile(0.50), 2),
        round(description_len.quantile(0.75), 2),
        int(description_len.max()),
        round(description_len.mean(), 2),
    ],
})

# description 길이 이상치 후보: 너무 짧은 경우(1~2자 이하), 너무 긴 경우(500자 이상)
description_short_df = df.loc[description_len <= 2, ["title", "description"]].copy()
description_short_df["description_len"] = description_len[description_len <= 2].values
description_long_df = df.loc[description_len >= 500, ["title", "description"]].copy()
description_long_df["description_len"] = description_len[description_len >= 500].values

# title 길이 이상치 확인
title_len = df["title"].fillna("").astype(str).str.len()
title_len_df = pd.DataFrame({
    "metric": ["count", "min", "25%", "50%", "75%", "max", "mean"],
    "value": [
        int(title_len.shape[0]),
        int(title_len.min()),
        round(title_len.quantile(0.25), 2),
        round(title_len.quantile(0.50), 2),
        round(title_len.quantile(0.75), 2),
        int(title_len.max()),
        round(title_len.mean(), 2),
    ],
})

title_short_df = df.loc[title_len <= 2, ["title", "description"]].copy()
title_short_df["title_len"] = title_len[title_len <= 2].values
title_long_df = df.loc[title_len >= 500, ["title", "description"]].copy()
title_long_df["title_len"] = title_len[title_len >= 500].values

print("[description length summary]")
display(description_len_df)
print(f"- description <= 2 chars: {len(description_short_df)}")
print(f"결측치 개수와 일치, discription이 존재하지 않는 강의")
print(f"\n- description >= 500 chars: {len(description_long_df)}")
display(description_short_df.head(10))
display(description_long_df.head(10))

print("[title length summary]")
display(title_len_df)
print(f"- title <= 2 chars: {len(title_short_df)}")
print(f"- title >= 500 chars: {len(title_long_df)}")
display(title_short_df.head(10))
display(title_long_df.head(10))


# description 최댓값, 최솟값 확인


[description length summary]


,metric,value
0,count,183.00
1,min,0.00
2,25%,80.50
3,50%,155.00
4,75%,356.00
5,max,4494.00
6,mean,412.83


- description <= 2 chars: 25
결측치 개수와 일치, discription이 존재하지 않는 강의

- description >= 500 chars: 40


,title,description,description_len
22,파이썬프로그래밍,NaN,0
26,AI가속기설계,NaN,0
36,손에잡히는데이터시각화,NaN,0
44,PR의이해,NaN,0
45,컴퓨팅적 사고,NaN,0
59,AI 반도체 공정,NaN,0
79,블록체인,NaN,0
87,Python을 이용한 금융 데이터 분석 및 적용,NaN,0
108,파이썬 프로그래밍,NaN,0
109,웹개발프로그래밍,NaN,0


,title,description,description_len
3,인공지능기초,"이 과목은 컴퓨터공학을 전공하는 학부 3학년 학생들을 대상으로 하며, 최신 인공지능...",549
33,대학생이라면 꼭 알아야 할 ICT 활용 특강,"이 강좌는 기존에 개별적으로 진행되었던 한글, 파워포인트, 엑셀, ChatGPT 활...",811
34,C프로그래밍 고급,"C프로그래밍은 SW전공자들이 1학년때 수강하는 기초과목이며, 2~3학년 SW전공과목...",523
113,논리와 IOT코딩,*논리와 IoT 코딩은 스마트 환경 기술이 기본이 되는 사물인터넷 환경을 직접 구축...,535
117,전자정부론,4차 산업혁명으로 촉발된 디지털 대전환 시대에 우리는 열린 사고를 바탕으로 국민이 ...,500
128,인공지능을 위한 기계학습 입문,인공지능을 위한 기계학습의 의미와 이를 가능하게 하는 기계학습 모델을 만드는 기술과...,1394
129,파이썬과 데이터마이닝,"본 수업은 빅데이터 연구를 하시려는 분, 혹은 본인의 적용 분야에서 데이터 마이닝을...",1150
130,소프트웨어적 사유,"강좌 소개 수업내용 목표 최근 많은 주목을 받고 있는 컴퓨팅 사고력, 즉 compu...",1716
131,21세기의 놀이하는 인간 : 컴퓨터게임 개론,KHU_컴퓨터게임 개론_강좌소개 “인간은 놀이를 한다. 인간의 놀이는 인간의 역사와...,2293
132,인공지능을 위한 R과 Python,강좌 소개 수업내용 인공지능 자료분석에 필요한 사용 예를 중심으로 R 과 Pytho...,1765


[title length summary]


,metric,value
0,count,183.00
1,min,4.00
2,25%,7.00
3,50%,10.00
4,75%,15.00
5,max,39.00
6,mean,12.08


- title <= 2 chars: 0
- title >= 500 chars: 0


,title,description,title_len


,title,description,title_len


### 이상치 및 데이터 품질 분석 결과 

1. `description`의 경우, 결측치가 존재하는 데이터가 있음을 파악
-> 약 25개로, 데이터베이스에서 제외하는 것으로 판단

2. `title`의 경우, 2자 이하 or 500자 이상의 극단적 데이터는 존재하지 않음. 

## 5. 정제 작업 수행

In [ ]:
# 5. 정제 작업 수행
# NOTE: 이전 단계에서 df가 로드되어 있어야 합니다. df가 없으면 NameError가 발생합니다.

import pandas as pd
from pathlib import Path
import unicodedata
from IPython.display import display

def normalize_text(value):
    return unicodedata.normalize("NFC", str(value)).strip()

before_rows = len(df)
before_cols = df.shape[1]

# 1) 불필요하거나 결측치가 너무 많은 컬럼 제거
#    사용자 요청의 "thunmbnail_url"은 실제 컬럼명 "thumbnail_url"로 반영합니다.
drop_columns = ["semester", "thumbnail_url", "syllabus_url", "view_count", "popular_score"]
existing_drop_columns = [col for col in drop_columns if col in df.columns]

rows_before_drop_columns = len(df)
df = df.drop(columns=existing_drop_columns)
rows_after_drop_columns = len(df)

# 2) description이 결측치인 행 삭제
rows_before_dropna = len(df)
df = df.dropna(subset=["description"])
rows_after_dropna = len(df)

# 3) main_category와 sub_category 기준 필터링
#    - main_category는 공학만 유지
#    - sub_category는 실제 분포에서 확인된 6개만 유지
allowed_sub_categories = {
    normalize_text("컴퓨터공학"),
    normalize_text("소프트웨어공학"),
    normalize_text("컴퓨터 · 통신"),
    normalize_text("컴퓨터과학"),
    normalize_text("정보통신공학"),
    normalize_text("정보과학"),
}

main_category_normalized = df["main_category"].map(normalize_text)
sub_category_normalized = df["sub_category"].map(normalize_text)
eng_category = normalize_text("공학")

rows_before_main_category = len(df)
df = df[main_category_normalized == eng_category].copy()
rows_after_main_category = len(df)

rows_before_sub_category = len(df)
df = df[sub_category_normalized.loc[df.index].isin(allowed_sub_categories)].copy()
rows_after_sub_category = len(df)

# 정제 결과 확인
after_rows = len(df)
after_cols = df.shape[1]
removed_rows = before_rows - after_rows
removed_cols = before_cols - after_cols

summary_df = pd.DataFrame({
    "metric": ["rows_before", "rows_after", "removed_rows", "cols_before", "cols_after", "removed_cols"],
    "value": [before_rows, after_rows, removed_rows, before_cols, after_cols, removed_cols],
})

step_summary_df = pd.DataFrame({
    "step": [
        "drop_columns",
        "dropna(description)",
        "filter main_category == 공학",
        "filter allowed sub_category",
    ],
    "rows_before": [
        rows_before_drop_columns,
        rows_before_dropna,
        rows_before_main_category,
        rows_before_sub_category,
    ],
    "rows_after": [
        rows_after_drop_columns,
        rows_after_dropna,
        rows_after_main_category,
        rows_after_sub_category,
    ],
})
step_summary_df["rows_removed"] = step_summary_df["rows_before"] - step_summary_df["rows_after"]

# 정제 후 결과를 CSV로 저장
# NOTE: notebook 실행 위치가 현재 폴더가 아니라면 저장 경로를 절대경로로 바꿔주세요.
output_path = Path("learning_resource_refactor.csv")
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("[Cleaning summary]")
display(summary_df)

print("[Rows removed by step]")
display(step_summary_df)

print("[Remaining columns]")
display(pd.DataFrame({"column": df.columns}))

print(f"[Saved] {output_path.resolve()}")

[Cleaning summary]


,metric,value
0,rows_before,0
1,rows_after,0
2,removed_rows,0
3,cols_before,11
4,cols_after,11
5,removed_cols,0


[Rows removed by step]


,step,rows_before,rows_after,rows_removed
0,drop_columns,0,0,0
1,dropna(description),0,0,0
2,filter main_category == ??,0,0,0
3,filter allowed sub_category,0,0,0


[Remaining columns]


,column
0,source_type
1,external_id
2,main_category
3,sub_category
4,title
5,description
6,provider_name
7,instructor_name
8,difficulty_level
9,url


[Saved] C:\Users\SSAFY\Desktop\PJT\WhereToGo_PJT\curriculum-data\raw\learning_resources\learning_resource_refactor.csv


In [25]:
display(df)

,source_type,external_id,main_category,sub_category,title,description,provider_name,instructor_name,difficulty_level,url,content_type


## 6. 정제 결과 검증